# PCA - Principal Component Analysis for Dimensionality Reduction

## Session 4: High-Dimensional Data Visualization

**Learning Objectives:**
1. Understand the curse of dimensionality
2. Implement PCA for dimensionality reduction
3. Interpret explained variance
4. Reduce MNIST from 784 to 2 dimensions
5. Apply PCA before clustering for speed improvements

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import time
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## Load MNIST Digits Dataset (8x8 version)

In [ ]:
# Load digits dataset (8x8 = 64 features)
digits = load_digits()
X = digits.data
y = digits.target

print(f'Dataset shape: {X.shape}')
print(f'Number of samples: {X.shape[0]}')
print(f'Number of features: {X.shape[1]}')
print(f'Number of classes: {len(np.unique(y))}')
print(f'Target labels: {np.unique(y)}')

In [ ]:
# Visualize sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for idx, ax in enumerate(axes.ravel()):
    ax.imshow(X[idx].reshape(8, 8), cmap='gray')
    ax.set_title(f'Label: {y[idx]}')
    ax.axis('off')
plt.suptitle('Sample Handwritten Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Standardize Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Data standardized: mean={X_scaled.mean():.3f}, std={X_scaled.std():.3f}')

## PCA: Explained Variance Analysis

In [ ]:
# Fit PCA with all components
pca_full = PCA()
pca_full.fit(X_scaled)

# Calculate cumulative explained variance
cumsum = np.cumsum(pca_full.explained_variance_ratio_)

# Plot explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Individual explained variance
ax1.bar(range(1, len(pca_full.explained_variance_ratio_) + 1), 
        pca_full.explained_variance_ratio_, color='steelblue', alpha=0.7)
ax1.set_xlabel('Principal Component', fontsize=12)
ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
ax1.set_title('Explained Variance by Component', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Cumulative explained variance
ax2.plot(range(1, len(cumsum) + 1), cumsum, 'b-', linewidth=2)
ax2.axhline(y=0.95, color='r', linestyle='--', label='95% variance')
ax2.axhline(y=0.90, color='orange', linestyle='--', label='90% variance')
ax2.set_xlabel('Number of Components', fontsize=12)
ax2.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax2.set_title('Cumulative Explained Variance', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find number of components for 95% variance
n_components_95 = np.argmax(cumsum >= 0.95) + 1
print(f'Components needed for 95% variance: {n_components_95}')
print(f'Original dimensions: {X.shape[1]}')
print(f'Reduction: {X.shape[1]} -> {n_components_95} ({n_components_95/X.shape[1]*100:.1f}%)']

## PCA: 2D Visualization

In [ ]:
# Reduce to 2D for visualization
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

print(f'2D PCA shape: {X_pca_2d.shape}')
print(f'PC1 explains: {pca_2d.explained_variance_ratio_[0]:.2%}')
print(f'PC2 explains: {pca_2d.explained_variance_ratio_[1]:.2%}')
print(f'Total: {sum(pca_2d.explained_variance_ratio_):.2%}')

In [ ]:
# Visualize in 2D
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='tab10', 
                     s=20, alpha=0.6, edgecolors='black', linewidth=0.5)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('MNIST Digits in 2D (PCA)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Digit', ticks=range(10))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## PCA: 3D Visualization

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Reduce to 3D
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(X_pca_3d[:, 0], X_pca_3d[:, 1], X_pca_3d[:, 2], 
                    c=y, cmap='tab10', s=20, alpha=0.6)
ax.set_xlabel(f'PC1 ({pca_3d.explained_variance_ratio_[0]:.1%})', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_3d.explained_variance_ratio_[1]:.1%})', fontsize=11)
ax.set_zlabel(f'PC3 ({pca_3d.explained_variance_ratio_[2]:.1%})', fontsize=11)
ax.set_title('MNIST Digits in 3D (PCA)', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Digit', shrink=0.5)
plt.tight_layout()
plt.show()

## PCA for Clustering Speed Improvement

In [ ]:
# Clustering on original data
print('Clustering on original 64-dimensional data...')
start = time.time()
kmeans_orig = KMeans(n_clusters=10, random_state=42, n_init=10)
labels_orig = kmeans_orig.fit_predict(X_scaled)
time_orig = time.time() - start
silhouette_orig = silhouette_score(X_scaled, labels_orig)
print(f'Time: {time_orig:.3f}s, Silhouette: {silhouette_orig:.4f}')

# Clustering on PCA-reduced data (20 components)
print('\nClustering on PCA-reduced data (20 components)...')
pca_20 = PCA(n_components=20)
X_pca_20 = pca_20.fit_transform(X_scaled)
start = time.time()
kmeans_pca = KMeans(n_clusters=10, random_state=42, n_init=10)
labels_pca = kmeans_pca.fit_predict(X_pca_20)
time_pca = time.time() - start
silhouette_pca = silhouette_score(X_pca_20, labels_pca)
print(f'Time: {time_pca:.3f}s, Silhouette: {silhouette_pca:.4f}')

print(f'\nSpeedup: {time_orig/time_pca:.2f}x faster with PCA!')

## Summary

**Key Takeaways:**
1. PCA successfully reduced 64D to 2D/3D for visualization
2. ~95% variance retained with ~{n_components_95} components (vs 64 original)
3. Digit clusters clearly visible in 2D/3D
4. Significant speed improvements for clustering
5. PCA preserves global structure well

**When to Use PCA:**
- High-dimensional data visualization
- Speed up algorithms (clustering, classification)
- Remove correlated features
- Noise reduction
- Feature extraction

**Next:** t-SNE for even better local structure preservation!